# 02 — Genomic Structure: 1001 Genomes × AraPheno Intersection

**Case Study A — Plant Intelligence Lab**

This notebook establishes the genomic population that can legitimately feed the first quantitative-genetics benchmark.

The modelling population is defined as

\[
\mathcal{A}_{\mathrm{model}}
=
\mathcal{A}_{\mathrm{phenotype}}
\cap
\mathcal{A}_{\mathrm{genomic}}.
\]

The notebook does four things before Model 1 is fitted:

1. verifies which AraPheno accessions have 1001 Genomes support;
2. loads the official 1135-accession imputed SNP matrix;
3. filters genomic markers transparently and computes the genomic relationship matrix;
4. characterizes population structure and constructs genotype-aware validation groups.

No predictive model is fitted here. The objective is to define the evidence base and the geometry of the genomic problem.


## Public genomic source

The 1001 Genomes Project completed its first major phase with **1,135 *Arabidopsis thaliana* genomes**. Its public Data Center distributes VCFs, pseudogenomes, and an imputed SNP matrix for the 1135-accession panel.

This notebook uses the official imputed SNP matrix archive when available locally:

`https://1001genomes.org/data/GMI-MPI/releases/current/SNP_matrix_imputed_hdf5/1001_SNP_MATRIX.tar.gz`

The archive is large (roughly 317 MB), so it is downloaded only when absent and is never committed to Git.


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import tarfile
import urllib.request

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ROOT = Path("..").resolve()
INTERIM = ROOT / "data" / "interim" / "case_study_a"
PROCESSED = ROOT / "data" / "processed" / "case_study_a"
GENOMIC_RAW = ROOT / "data" / "raw" / "1001genomes" / "1135"
RESULTS = ROOT / "reports" / "results"
FIGURES = ROOT / "reports" / "figures"

for path in [INTERIM, PROCESSED, GENOMIC_RAW, RESULTS, FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

SNP_ARCHIVE_URL = (
    "https://1001genomes.org/data/GMI-MPI/releases/current/"
    "SNP_matrix_imputed_hdf5/1001_SNP_MATRIX.tar.gz"
)
ARCHIVE = GENOMIC_RAW / "1001_SNP_MATRIX.tar.gz"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


## 1. Load the phenotype accession set created in Notebook 01

Notebook 01 writes accession-level phenotype summaries into `data/interim/case_study_a/`. This notebook refuses to invent an accession set if that output is absent.


In [ ]:
candidate_files = [
    INTERIM / "accession_summary.csv",
    INTERIM / "accession_phenotype_summary.csv",
    PROCESSED / "accession_summary.csv",
]

phenotype_path = next((p for p in candidate_files if p.exists()), None)

if phenotype_path is None:
    raise FileNotFoundError(
        "Run notebooks/01_data_discovery.ipynb first. "
        "No accession-level phenotype summary was found."
    )

phenotype = pd.read_csv(phenotype_path)
if "accession_id" not in phenotype.columns:
    raise KeyError(f"{phenotype_path} does not contain accession_id.")

phenotype["accession_id"] = phenotype["accession_id"].astype(str)
phenotype_accessions = sorted(phenotype["accession_id"].dropna().unique())

print("Phenotype source:", phenotype_path)
print("Unique phenotype accessions:", len(phenotype_accessions))


## 2. Acquire the official 1135-accession SNP matrix

The download is deterministic and stored under `data/raw/1001genomes/1135/`. Large raw files remain local because the repository should contain reproducible acquisition logic rather than redistributed genomic archives.


In [ ]:
def sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

if not ARCHIVE.exists():
    print(f"Downloading {SNP_ARCHIVE_URL}")
    urllib.request.urlretrieve(SNP_ARCHIVE_URL, ARCHIVE)
else:
    print("Using existing archive:", ARCHIVE)

print("Archive size (MB):", round(ARCHIVE.stat().st_size / 1024**2, 2))
print("SHA256:", sha256(ARCHIVE))


In [ ]:
extract_dir = GENOMIC_RAW / "SNP_matrix_imputed_hdf5"
extract_dir.mkdir(parents=True, exist_ok=True)

h5_candidates = list(extract_dir.rglob("*.h5")) + list(extract_dir.rglob("*.hdf5"))

if not h5_candidates:
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(extract_dir)
    h5_candidates = list(extract_dir.rglob("*.h5")) + list(extract_dir.rglob("*.hdf5"))

if not h5_candidates:
    raise FileNotFoundError("No HDF5 file was found after extracting the official SNP archive.")

print("HDF5 candidates:")
for p in h5_candidates:
    print(" -", p.relative_to(ROOT))


## 3. Inspect the HDF5 structure before assuming a schema

The official matrix format has changed across releases. The code therefore inventories groups and datasets first and selects candidate accession, marker-position, and genotype arrays by structure rather than silently assuming names.


In [ ]:
def h5_inventory(path: Path) -> pd.DataFrame:
    rows = []
    with h5py.File(path, "r") as h5:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                rows.append({
                    "name": name,
                    "shape": tuple(obj.shape),
                    "dtype": str(obj.dtype),
                    "ndim": obj.ndim,
                })
        h5.visititems(visitor)
    return pd.DataFrame(rows)

inventories = []
for p in h5_candidates:
    inv = h5_inventory(p)
    inv.insert(0, "file", str(p.relative_to(ROOT)))
    inventories.append(inv)

inventory = pd.concat(inventories, ignore_index=True)
inventory.sort_values(["file", "ndim", "name"]).head(50)


In [ ]:
# Candidate genotype matrices are 2D numeric arrays with one dimension close to the 1135 accession panel.
matrix_candidates = inventory[
    (inventory["ndim"] == 2)
    & inventory["dtype"].str.contains(r"int|float|bool", case=False, regex=True)
] .copy()

matrix_candidates


## 4. Resolve accession identifiers and genotype matrix orientation

The accession dimension must match a 1D identifier dataset. The resolver below searches for identifier-like arrays and verifies lengths against candidate genotype matrices. If more than one plausible pairing remains, the notebook stops and requires explicit selection rather than guessing.


In [ ]:
def decode_1d(arr):
    out = []
    for x in arr:
        if isinstance(x, (bytes, np.bytes_)):
            out.append(x.decode("utf-8"))
        else:
            out.append(str(x))
    return np.asarray(out, dtype=object)

one_d = inventory[inventory["ndim"] == 1].copy()

pairs = []
for _, m in matrix_candidates.iterrows():
    mshape = tuple(m["shape"])
    for _, d in one_d.iterrows():
        dlen = d["shape"][0]
        if dlen in mshape:
            score = 0
            name = d["name"].lower()
            if any(k in name for k in ["accession", "sample", "strain", "ecotype", "id"]):
                score += 10
            if 1000 <= dlen <= 2500:
                score += 5
            pairs.append({
                "matrix_file": m["file"],
                "matrix_name": m["name"],
                "matrix_shape": mshape,
                "id_file": d["file"],
                "id_name": d["name"],
                "id_length": dlen,
                "score": score,
            })

pair_table = pd.DataFrame(pairs).sort_values(["score", "id_length"], ascending=[False, True])
pair_table.head(20)


In [ ]:
if pair_table.empty:
    raise RuntimeError("No accession-ID / genotype-matrix pairing could be inferred from the HDF5 inventory.")

best = pair_table.iloc[0]
top_score = best["score"]
n_top = (pair_table["score"] == top_score).sum()

if n_top > 1:
    print("Multiple top-scoring candidates exist. Inspect pair_table before proceeding.")
    display(pair_table.head(20))
    raise RuntimeError("Ambiguous HDF5 schema: explicit selection required.")

matrix_path = ROOT / best["matrix_file"]
id_path = ROOT / best["id_file"]

with h5py.File(matrix_path, "r") as h5m, h5py.File(id_path, "r") as h5i:
    G_raw = np.asarray(h5m[best["matrix_name"]])
    accession_ids = decode_1d(np.asarray(h5i[best["id_name"]]))

if G_raw.shape[0] == len(accession_ids):
    G = G_raw
elif G_raw.shape[1] == len(accession_ids):
    G = G_raw.T
else:
    raise RuntimeError("Resolved accession length does not match genotype matrix.")

print("Resolved genotype matrix:", G.shape)
print("Resolved accession vector:", accession_ids.shape)


## 5. Construct the genotype–phenotype intersection

Only accessions with both phenotype evidence and genomic representation belong to the first modelling population.


In [ ]:
genomic_accessions = pd.Index(accession_ids.astype(str))
phenotype_index = pd.Index(phenotype_accessions)

model_accessions = phenotype_index.intersection(genomic_accessions)

intersection_summary = pd.DataFrame({
    "population": [
        "phenotype accessions",
        "genomic accessions",
        "matched modelling accessions",
    ],
    "n": [
        len(phenotype_index),
        len(genomic_accessions),
        len(model_accessions),
    ],
})

intersection_summary


In [ ]:
if len(model_accessions) < 30:
    raise RuntimeError(
        f"Only {len(model_accessions)} matched accessions were found. "
        "Check identifier compatibility before modelling."
    )

accession_to_row = {str(a): i for i, a in enumerate(accession_ids)}
row_idx = np.array([accession_to_row[str(a)] for a in model_accessions], dtype=int)
G_model = np.asarray(G[row_idx, :], dtype=float)

print("Modelling genotype matrix before marker QC:", G_model.shape)


## 6. Marker quality control

The initial genomic benchmark removes markers that carry no usable information in the matched population. The default filters are intentionally transparent:

- remove markers with excessive missingness;
- remove monomorphic markers;
- remove markers below a minimum minor-allele frequency.

The exact thresholds are parameters and must be reported with every model result.


In [ ]:
MAX_MISSING = 0.10
MIN_MAF = 0.05

# Treat common negative missing codes as NaN.
G_model[G_model < 0] = np.nan

missing_rate = np.mean(np.isnan(G_model), axis=0)
keep_missing = missing_rate <= MAX_MISSING
G_qc = G_model[:, keep_missing]

# Mean-impute only after the missingness filter.
marker_means = np.nanmean(G_qc, axis=0)
nan_rows, nan_cols = np.where(np.isnan(G_qc))
G_qc[nan_rows, nan_cols] = marker_means[nan_cols]

print("Genotype range after missing-code handling:", float(np.nanmin(G_qc)), float(np.nanmax(G_qc)))

if np.nanmax(G_qc) <= 1.0:
    allele_freq = G_qc.mean(axis=0)
else:
    allele_freq = G_qc.mean(axis=0) / 2.0

maf = np.minimum(allele_freq, 1.0 - allele_freq)
variance = G_qc.var(axis=0)

keep_informative = (variance > 0) & (maf >= MIN_MAF)
G_qc = G_qc[:, keep_informative]

qc_summary = pd.DataFrame({
    "stage": [
        "raw matched markers",
        "after missingness filter",
        "after MAF + variance filters",
    ],
    "p": [
        G_model.shape[1],
        int(keep_missing.sum()),
        G_qc.shape[1],
    ],
})

qc_summary


## 7. Genomic relationship matrix

For centered genotype matrix \(\mathbf{M}\), a VanRaden-style genomic relationship matrix is

\[
\mathbf{K}
=
\frac{\mathbf{M}\mathbf{M}^{\top}}
{2\sum_{j=1}^{p} p_j(1-p_j)}.
\]

This matrix is the covariance geometry used by the GBLUP baseline in Model 1.


In [ ]:
# Recompute allele frequencies after QC.
if np.nanmax(G_qc) <= 1.0:
    p_j = G_qc.mean(axis=0)
    M = G_qc - p_j
    denom = np.sum(p_j * (1 - p_j))
else:
    p_j = G_qc.mean(axis=0) / 2.0
    M = G_qc - 2.0 * p_j
    denom = 2.0 * np.sum(p_j * (1 - p_j))

if denom <= 0:
    raise RuntimeError("Non-positive GRM denominator after marker QC.")

K = (M @ M.T) / denom
K = (K + K.T) / 2.0

print("K shape:", K.shape)
print("K diagonal mean:", float(np.mean(np.diag(K))))
print("K eigenvalue minimum:", float(np.linalg.eigvalsh(K).min()))


## 8. Population structure

Principal components summarize the dominant axes of genomic variation. These components are useful both for interpretation and for designing validation splits that are harder than random holdouts.


In [ ]:
# PCA on samples; markers are standardized to equalize scale after QC.
X = StandardScaler(with_mean=True, with_std=True).fit_transform(G_qc)
n_components = min(10, X.shape[0] - 1, X.shape[1])
pca = PCA(n_components=n_components, random_state=42)
scores = pca.fit_transform(X)

pca_table = pd.DataFrame(
    scores,
    index=model_accessions,
    columns=[f"PC{i+1}" for i in range(scores.shape[1])]
)
pca_table.index.name = "accession_id"

explained = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(scores.shape[1])],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
})

explained


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(pca_table["PC1"], pca_table["PC2"], s=30, alpha=0.8)
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% variance)")
ax.set_title("Case Study A — genomic structure of matched accessions")
fig.tight_layout()

pca_fig = FIGURES / "case_study_a_genomic_pca.png"
fig.savefig(pca_fig, dpi=200, bbox_inches="tight")
plt.show()


## 9. Genotype-aware validation groups

A random split can place genetically similar material on both sides of the evaluation boundary. This notebook therefore creates genomic clusters from principal components. Model 1 will use these groups to construct held-out folds.

The number of clusters is not a biological truth claim. It is a validation device whose stability must be inspected.


In [ ]:
N_CLUSTERS = min(5, max(2, len(model_accessions) // 20))

cluster_features = scores[:, : min(5, scores.shape[1])]
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=50)
clusters = kmeans.fit_predict(cluster_features)

validation_groups = pd.DataFrame({
    "accession_id": model_accessions.astype(str),
    "genomic_cluster": clusters,
})

validation_groups["genomic_cluster"].value_counts().sort_index()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
for cluster, df in (
    pca_table.reset_index()
    .merge(validation_groups, on="accession_id")
    .groupby("genomic_cluster")
):
    ax.scatter(df["PC1"], df["PC2"], s=34, alpha=0.85, label=f"cluster {cluster}")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Genotype-aware validation groups")
ax.legend(frameon=False)
fig.tight_layout()

cluster_fig = FIGURES / "case_study_a_genomic_validation_groups.png"
fig.savefig(cluster_fig, dpi=200, bbox_inches="tight")
plt.show()


## 10. Persist the Model 1 inputs

The modelling population, filtered genotype matrix, relationship matrix, PCA representation, and validation groups are written explicitly. Model 1 must use these exact outputs so that every downstream comparison is made on the same genomic evidence base.


In [ ]:
np.save(PROCESSED / "case_study_a_genotype_matrix.npy", G_qc)
np.save(PROCESSED / "case_study_a_grm.npy", K)

pd.DataFrame({"accession_id": model_accessions.astype(str)}).to_csv(
    PROCESSED / "case_study_a_model_accessions.csv", index=False
)
pca_table.reset_index().to_csv(
    PROCESSED / "case_study_a_genomic_pca.csv", index=False
)
validation_groups.to_csv(
    PROCESSED / "case_study_a_validation_groups.csv", index=False
)

intersection_summary.to_csv(
    RESULTS / "case_study_a_genotype_phenotype_intersection.csv", index=False
)
qc_summary.to_csv(
    RESULTS / "case_study_a_marker_qc.csv", index=False
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": SNP_ARCHIVE_URL,
    "archive_sha256": sha256(ARCHIVE),
    "n_phenotype_accessions": int(len(phenotype_index)),
    "n_genomic_accessions": int(len(genomic_accessions)),
    "n_model_accessions": int(len(model_accessions)),
    "p_after_qc": int(G_qc.shape[1]),
    "max_missing": MAX_MISSING,
    "min_maf": MIN_MAF,
    "n_validation_clusters": int(N_CLUSTERS),
}

(PROCESSED / "case_study_a_genomic_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

manifest


## 11. Readiness for Model 1

Model 1 is allowed to begin only after this notebook establishes:

- a non-trivial genotype–phenotype accession intersection;
- a documented marker-filtering pipeline;
- a valid genomic relationship matrix;
- visible population structure;
- genotype-aware validation groups;
- persisted modelling inputs and provenance.

The next notebook is:

`03_genomic_prediction.ipynb`

Its first benchmark is the classical quantitative-genetics model

\[
\mathbf{y}
=
\mathbf{X}\boldsymbol{\beta}
+
\mathbf{Z}\mathbf{u}
+
\boldsymbol{\varepsilon},
\qquad
\mathbf{u}\sim\mathcal{N}
\left(
\mathbf{0},
\sigma_g^2\mathbf{K}
\right).
\]

Only after this baseline is evaluated under genotype-aware validation should high-dimensional machine-learning models be compared against it.
